# Use oracle to synthesis roles for each table in the spider database

### 1. Setup LLM Oracle instance and assign roles to tables

In [15]:
import sys
sys.path.append('/home/feiy/Role-SQL-benchmark')

from llm_oracle import Oracle
import glob
import os
from dotenv import load_dotenv
import time

load_dotenv()

True

In [16]:
# DeepSeek demo
DEEPSEEK_API_KEY = os.getenv('DEEPSEEK_API_KEY')
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

if not DEEPSEEK_API_KEY:
    print("Warning: Cannot find DEEPSEEK_API_KEY in environment")
    print("Please set it in the .env file or environment")
if not OPENAI_API_KEY:
    print("Warning: Cannot find OPENAI_API_KEY in environment")
    print("Please set it in the .env file or environment")


1. demo test

In [3]:
if not DEEPSEEK_API_KEY:
    print("Warning: DEEPSEEK_API_KEY not found in environment variables")
    print("Please set it in your .env file or environment")
else:
    # Create Oracle instance with DeepSeek model
    oracle = Oracle(model="deepseek-chat", apikey=DEEPSEEK_API_KEY)
    
    # Test the model
    test_response = oracle.query(
        prompt_sys="You are a helpful assistant.",
        prompt_user="Say Hi.",
        temp=0.7,
        top_p=0.9
    )
    
    print("Model Response:")
    print("="*50)
    print(test_response['answer'])

Model Response:
Hi! How can I help you today? 😊


2. prompt caching test

In [ ]:
# Create two identical requests to test caching
prompt_sys = """You are a helpful assistant. You should:
1. Be concise and clear in your responses
2. Always strive to provide accurate information
3. Maintain a professional and friendly tone
4. Use appropriate formatting when needed
5. Ask for clarification if something is unclear"""

prompt_user = """Please introduce yourself and tell me about your capabilities.
Make sure to mention:
1. Your name
2. Your main areas of expertise
3. How you can help users
4. Any limitations users should be aware of"""

In [ ]:
def make_query(prompt_sys, prompt_user):
    """Execute query and return response"""
    response = oracle.query(
        prompt_sys=prompt_sys,
        prompt_user=prompt_user,
    )
    # if response.get('answer'):
    #     print(f"Response: {response['answer']}")
    return response

def print_cache_stats(response, label=None):
    """Print cache statistics for a response"""
    if label:
        print(f"\n=== {label} Statistics ===")
    
    usage = response.get('usage', {})
    prompt_details = usage.get('prompt_tokens_details', None)
    
    print("\nToken Statistics:")
    print(f"  Prompt tokens: {usage.get('prompt_tokens', 0)}")
    print(f"  Completion tokens: {usage.get('completion_tokens', 0)}")
    print(f"  Total tokens: {usage.get('total_tokens', 0)}")
    
    print("\nCache Statistics:")
    if prompt_details:
        if isinstance(prompt_details, str):
            print(f"  Raw info: {prompt_details}")
        else:
            cached = getattr(prompt_details, 'cached_tokens', 0)
            non_cached = usage.get('prompt_tokens', 0) - cached
            print(f"  Cached tokens: {cached}")
            print(f"  Non-cached tokens: {non_cached}")
            if cached > 0:
                print(f"  Cache hit rate: {(cached / usage.get('prompt_tokens', 1)) * 100:.1f}%")

In [ ]:
response1 = make_query(prompt_sys, prompt_user)
response2 = make_query(prompt_sys, prompt_user)
response3 = make_query(prompt_sys, prompt_user)

print_cache_stats(response1, "First Call")
print_cache_stats(response2, "Second Call")
print_cache_stats(response3, "Third Call")


=== First Call Statistics ===

Token Statistics:
  Prompt tokens: 103
  Completion tokens: 237
  Total tokens: 340

Cache Statistics:
  Cached tokens: 64
  Non-cached tokens: 39
  Cache hit rate: 62.1%

=== Second Call Statistics ===

Token Statistics:
  Prompt tokens: 103
  Completion tokens: 246
  Total tokens: 349

Cache Statistics:
  Cached tokens: 64
  Non-cached tokens: 39
  Cache hit rate: 62.1%

=== Third Call Statistics ===

Token Statistics:
  Prompt tokens: 103
  Completion tokens: 273
  Total tokens: 376

Cache Statistics:
  Cached tokens: 64
  Non-cached tokens: 39
  Cache hit rate: 62.1%


In [ ]:
# Example: create Oracle instance (model and api_key should be set according to your environment)
MODEL_NAME = 'gpt-4o'  # or any supported model
API_KEY = 'your_api_key_here'  # replace with your actual key
oracle_instance = Oracle(MODEL_NAME, API_KEY)
print(f"Oracle instance created for model: {MODEL_NAME}")

In [ ]:
# Example: get all table files (input path can be customized)
table_sql_dir = '/path/to/your/tables/'  # TODO: set your actual path
table_files = glob.glob(os.path.join(table_sql_dir, '*.sql'))
print(f"Found {len(table_files)} tables.")

# Example workflow: assign roles to each table (prompt design TBD)
def assign_role_to_table(table_file, oracle_instance):
    # Read table schema or content
    with open(table_file, 'r') as f:
        table_content = f.read()
    # TODO: Design prompt for role assignment
    prompt = f"You are a database expert. Please assign a possible role to the following table based on its schema and content.\nTable:\n{table_content}"
    result = oracle_instance.query('You are a helpful assistant.', prompt)
    return result['answer']

# Example: assign roles for all tables
table_roles = {}
for table_file in table_files:
    table_name = os.path.basename(table_file)
    role = assign_role_to_table(table_file, oracle_instance)
    table_roles[table_name] = role
    print(f"Table: {table_name}, Assigned Role: {role}")